In [13]:
import numpy as np
from tqdm import tqdm, trange
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from joblib import Parallel, delayed
import multiprocessing

In [5]:

def DKRL_dual(Z, X, y, r, penalty, tol, T):
    p, q, N = Z.shape[1], X.shape[1], Z.shape[0]
    Ut = np.random.normal(0, 1, (p, r))
    Vt = np.random.normal(0, 1, (q, r))
    Utt = Ut.copy()
    Vtt = Vt.copy()
    I_p = np.eye(p)
    I_q = np.eye(q)
    alpha = 1.2
    prev_y_pred = (Z @ Utt) * (X @ Vtt)
    y_pred_total = prev_y_pred.sum(axis=1)

    for _ in range(int(T)):
        hhat = X @ Vtt
        ghat = Z @ Utt

        for i in range(r):
            y_pred_i = ghat[:, i] * hhat[:, i]
            y_residual = y - (y_pred_total - y_pred_i)
            DZ = Z * hhat[:, i][:, np.newaxis]
            A = DZ.T @ DZ + penalty * I_p
            b = DZ.T @ y_residual
            update = np.linalg.solve(A, b)
            Utt[:, i] = alpha * update + (1 - alpha) * Utt[:, i]

        ghat = Z @ Utt
        for i in range(r):
            hhat = X @ Vtt
            y_pred_i = ghat[:, i] * hhat[:, i]
            y_residual = y - (ghat * hhat).sum(axis=1) + y_pred_i
            DX = X * ghat[:, i][:, np.newaxis]
            A = DX.T @ DX + penalty * I_q
            b = DX.T @ y_residual
            update = np.linalg.solve(A, b)
            Vtt[:, i] = alpha * update + (1 - alpha) * Vtt[:, i]

        ghat = Z @ Utt
        hhat = X @ Vtt
        y_pred_new = (ghat * hhat).sum(axis=1)

        normU = np.linalg.norm(Ut)
        normV = np.linalg.norm(Vt)
        delta_U = np.linalg.norm(Utt - Ut) / (normU + 1e-4)
        delta_V = np.linalg.norm(Vtt - Vt) / (normV + 1e-3)
        if delta_U < tol and delta_V < tol:
            break

        Ut[:, :] = Utt
        Vt[:, :] = Vtt
        y_pred_total = y_pred_new

    return Utt, Vtt, y_pred_total

def DKRL_pred_dual(U, V, Z, X):
    y_pred = (Z @ U) * (X @ V)
    return y_pred.sum(axis=1)

In [ ]:
def run_simulation(Z, X, Theta, penalty, N, r, tol, T):
    idx_Z = np.random.choice(Z.shape[0], N, replace=True)
    idx_X = np.random.choice(X.shape[0], N, replace=True)
    Zs, Xs = Z[idx_Z], X[idx_X]
    y = np.diag(Zs @ Theta @ Xs.T)

    N_train = int(0.9 * N)
    idx_train = np.random.choice(N, N_train, replace=False)
    idx_test = np.setdiff1d(np.arange(N), idx_train)
    Z_train, X_train, y_train = Zs[idx_train], Xs[idx_train], y[idx_train]
    Z_test, X_test, y_test = Zs[idx_test], Xs[idx_test], y[idx_test]

    output = {}

    # DKRL (dual)
    U, V, y_hat = DKRL_dual(Z_train, X_train, y_train, r=7, penalty=penalty, tol=tol, T=T)
    y_pred = DKRL_pred_dual(U, V, Z_test, X_test)
    output["DKRL"] = [np.sqrt(mean_squared_error(y_train, y_hat)), np.sqrt(mean_squared_error(y_test, y_pred))]

    # Lasso
    output["Lasso_Z"] = [
        np.sqrt(mean_squared_error(y_train, Lasso(alpha=0.01, max_iter=1000).fit(Z_train, y_train).predict(Z_train))),
        np.sqrt(mean_squared_error(y_test, Lasso(alpha=0.01).fit(Z_train, y_train).predict(Z_test)))
    ]
    output["Lasso_X"] = [
        np.sqrt(mean_squared_error(y_train, Lasso(alpha=0.01, max_iter=1000).fit(X_train, y_train).predict(X_train))),
        np.sqrt(mean_squared_error(y_test, Lasso(alpha=0.01).fit(X_train, y_train).predict(X_test)))
    ]
    output["Lasso_ZX"] = [
        np.sqrt(mean_squared_error(y_train, Lasso(alpha=0.01, max_iter=1000).fit(np.hstack([Z_train, X_train]), y_train).predict(np.hstack([Z_train, X_train])))),
        np.sqrt(mean_squared_error(y_test, Lasso(alpha=0.01).fit(np.hstack([Z_train, X_train]), y_train).predict(np.hstack([Z_test, X_test]))))
    ]

    # Random Forest
    output["RF_Z"] = [
        np.sqrt(mean_squared_error(y_train, RandomForestRegressor().fit(Z_train, y_train).predict(Z_train))),
        np.sqrt(mean_squared_error(y_test, RandomForestRegressor().fit(Z_train, y_train).predict(Z_test)))
    ]
    output["RF_X"] = [
        np.sqrt(mean_squared_error(y_train, RandomForestRegressor().fit(X_train, y_train).predict(X_train))),
        np.sqrt(mean_squared_error(y_test, RandomForestRegressor().fit(X_train, y_train).predict(X_test)))
    ]
    output["RF_ZX"] = [
        np.sqrt(mean_squared_error(y_train, RandomForestRegressor().fit(np.hstack([Z_train, X_train]), y_train).predict(np.hstack([Z_train, X_train])))),
        np.sqrt(mean_squared_error(y_test, RandomForestRegressor().fit(np.hstack([Z_train, X_train]), y_train).predict(np.hstack([Z_test, X_test]))))
    ]

    # Feedforward NN
    output["FNN_Z"] = [
        np.sqrt(mean_squared_error(y_train, MLPRegressor(hidden_layer_sizes=(100,), max_iter=500).fit(Z_train, y_train).predict(Z_train))),
        np.sqrt(mean_squared_error(y_test, MLPRegressor(hidden_layer_sizes=(100,), max_iter=500).fit(Z_train, y_train).predict(Z_test)))
    ]
    output["FNN_X"] = [
        np.sqrt(mean_squared_error(y_train, MLPRegressor(hidden_layer_sizes=(100,), max_iter=500).fit(X_train, y_train).predict(X_train))),
        np.sqrt(mean_squared_error(y_test, MLPRegressor(hidden_layer_sizes=(100,), max_iter=500).fit(X_train, y_train).predict(X_test)))
    ]
    output["FNN_ZX"] = [
        np.sqrt(mean_squared_error(y_train, MLPRegressor(hidden_layer_sizes=(100,), max_iter=500).fit(np.hstack([Z_train, X_train]), y_train).predict(np.hstack([Z_train, X_train])))),
        np.sqrt(mean_squared_error(y_test, MLPRegressor(hidden_layer_sizes=(100,), max_iter=500).fit(np.hstack([Z_train, X_train]), y_train).predict(np.hstack([Z_test, X_test]))))
    ]

    return output

    
# Settings
MC = 1
penalty = 0.01
q_list = [5, 4, 3, 2, 1]
r = 20
tol, T = 1e-4, 10000
n_theta = 5

# Load and normalize data
Z = np.load("X_news_embeddings_unique.npy")
X = np.load("X_user_features_unique.npy")
Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)
X = X / np.linalg.norm(X, axis=1, keepdims=True)

p, q = Z.shape[1], X.shape[1]
N = 5000
np.random.seed(2025)

print(f"Available CPU cores: {multiprocessing.cpu_count()}")


results = {q_decay: {
    "DKRL": [], "Lasso_Z": [], "Lasso_X": [], "Lasso_ZX": [],
    "RF_Z": [], "RF_X": [], "RF_ZX": [],
    "FNN_Z": [], "FNN_X": [], "FNN_ZX": []
} for q_decay in q_list}

for q_decay in q_list:
    eigs = np.concatenate([np.ones(5), np.array([i**(-q_decay) for i in range(6, r + 1)])])
    Sigma = np.diag(eigs)

    for theta_iter in trange(n_theta, desc=f"q={q_decay} - Theta loop"):
        P = np.random.normal(0, 1, (p, q))
        L, _, Rt = np.linalg.svd(P, full_matrices=False)
        Lr, Rr = L[:, :r], Rt[:r, :].T
        Theta = Lr @ Sigma @ Rr.T

        outputs = Parallel(n_jobs=10)(
            delayed(run_simulation)(Z, X, Theta, penalty, N, r, tol, T)
            for _ in trange(MC, desc=f"MC simulations for q={q_decay}, theta={theta_iter}", leave=False)
        )

        for res in outputs:
            for method in res:
                results[q_decay][method].append(res[method])

np.save("MC_simulation_results_fixed_penalty_with_sigma_and_baselines.npy", results)


/var/folders/jh/3h_6szkj6jv547pqqs09_tkh0000gq/T/ipykernel_17369/4234815207.py:12: RuntimeWarning: invalid value encountered in divide
  Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)


Available CPU cores: 16


q=5 - Theta loop:   0%|          | 0/5 [02:48<?, ?it/s]


ValueError: Input contains NaN.